# 🚀 SMIRK Demo on Google Colab (Fast Setup)

Uses Python 3.10 conda environment with **prebuilt PyTorch3D wheels** (installs in seconds!)

---

## ⚙️ Step 1: Enable GPU

**Runtime → Change runtime type → GPU (T4)**

In [ ]:
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
else:
    print("❌ Enable GPU first!")

## 📦 Step 2: Install Miniconda & Create Python 3.10 Environment

In [ ]:
# Install Miniconda
!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O miniconda.sh
!bash miniconda.sh -b -f -p /usr/local/miniconda
!rm miniconda.sh

# Add to PATH
import os
os.environ['PATH'] = '/usr/local/miniconda/bin:' + os.environ['PATH']

print("✅ Miniconda installed!")

In [ ]:
# Create Python 3.10 environment
!conda create -n p3d python=3.10 -y
print("✅ Python 3.10 environment created!")

In [ ]:
%%bash
source /usr/local/miniconda/bin/activate p3d

# Install PyTorch with CUDA 12.1
pip install torch==2.4.1+cu121 torchvision --index-url https://download.pytorch.org/whl/cu121

# Install PyTorch3D prebuilt wheel (FAST!)
pip install pytorch3d -f https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/py310_cu121_pyt241/download.html

python -c "import pytorch3d; print(f'✅ PyTorch3D {pytorch3d.__version__}')"

## 📥 Step 3: Clone Repository & Install Dependencies

In [ ]:
!git clone https://github.com/chinnu2534/smirk_ccn.git
%cd smirk_ccn

In [ ]:
%%bash
source /usr/local/miniconda/bin/activate p3d
cd smirk_ccn

pip install -q albumentations omegaconf scikit-learn scikit-image timm tqdm chumpy gdown pytorch_lightning
pip install -q opencv-python opencv-contrib-python mediapipe

echo "✅ Dependencies installed!"

## 📥 Step 4: Download Models

In [ ]:
%%bash
source /usr/local/miniconda/bin/activate p3d
cd smirk_ccn
bash quick_install.sh
echo "✅ Models downloaded!"

## 🖼️ Step 5: Upload Test Image

In [ ]:
from google.colab import files
import shutil, os

os.makedirs('samples', exist_ok=True)
print("Upload a face image:")
uploaded = files.upload()

for f in uploaded.keys():
    shutil.move(f, f'samples/{f}')
    test_image = f'samples/{f}'
    print(f"✅ Saved: {test_image}")
    break

from IPython.display import Image, display
display(Image(filename=test_image, width=300))

## 🎯 Step 6: Run Demo

In [ ]:
%%bash -s "$test_image"
source /usr/local/miniconda/bin/activate p3d
cd smirk_ccn
python demo.py --input_path "$1" --out_path results/ --checkpoint pretrained_models/SMIRK_em1.pt --crop
echo "✅ Done!"

## 📊 Step 7: View Results

In [ ]:
from IPython.display import Image, display
import os

results_dir = 'results'
if os.path.exists(results_dir):
    for f in sorted(os.listdir(results_dir)):
        if f.endswith(('.png', '.jpg')):
            print(f"📸 {f}")
            display(Image(filename=os.path.join(results_dir, f), width=800))

## 💾 Download Results

In [ ]:
import shutil
from google.colab import files
if os.path.exists('results'):
    shutil.make_archive('results', 'zip', 'results')
    files.download('results.zip')